In [ ]:
import mlflow

mlflow.set_experiment("Random Forest")

<Experiment: artifact_location='file:///home/sarah/code/forth_year/data_science/project/walmart-sales-classification/src/models/mlruns/345543488712145240', creation_time=1777473867656, experiment_id='345543488712145240', last_update_time=1777473867656, lifecycle_stage='active', name='Random Forest', tags={}>

In [ ]:
import pandas as pd
import numpy as np

# Modelling
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm

# Tree Visualisation
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

In [ ]:
train_df = pd.read_csv('../../data/model_ready/train.csv')
test_df = pd.read_csv('../../data/model_ready/test.csv')

In [ ]:
features_selected = [
    "Size",
    "Size",
    "Store",
    "Dept",
    "CPI",
    "DeptFrequency",
    "Week_cos",
    "IsPreHoliday",
    "Week_sin",
    "Fuel_Price",
    "ConsumerConfRatio",
    "AvgMarkDownAmount"
    ]

In [ ]:
X_train = train_df.iloc[:, : -1]
IsHoliday = test_df['IsHoliday']
y_train = train_df.iloc[:, -1].squeeze()
X_test = test_df.iloc[:, :-1]
y_test = test_df.iloc[:, -1]

In [ ]:
X_train = X_train[features_selected]
X_test = X_test[features_selected]

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
import mlflow

# Enable autologging for scikit-learn
mlflow.sklearn.autolog()



2026/04/29 19:00:03 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.8.0. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'bootstrap': [True]
}

# 3. Initialize Grid Search
# cv=kf uses your 5-fold cross-validation strategy
# n_jobs=-1 uses all CPU cores
# verbose=2 acts as your "loading bar" in the console
rf = RandomForestClassifier(random_state=42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)



In [ ]:
import joblib
import os
import mlflow
from sklearn.ensemble import RandomForestClassifier


MODEL_PATH = "best_rf_model.pkl"

def load_or_train_model(X, y, params):
    """Checks for existing model, otherwise trains a new one."""
    if os.path.exists(MODEL_PATH):
        print(f"--- Found existing model at {MODEL_PATH}. Loading... ---")
        model = joblib.load(MODEL_PATH)
    else:
        print("--- No local model found. Training from scratch... ---")
        model = RandomForestClassifier(**params)
        model.fit(X, y)
        
        # Save to local file
        joblib.dump(model, MODEL_PATH)
        print(f"--- Model saved to {MODEL_PATH} ---")
        
    return model


In [ ]:
import os
import joblib
import mlflow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

MODEL_PATH = "best_rf_grid_model.pkl"

if os.path.exists(MODEL_PATH):
    print(f"Loading existing best model from {MODEL_PATH}...")
    best_rf = joblib.load(MODEL_PATH)
    best_params = best_rf.get_params()
    best_score = best_rf.score(X_test, y_test)
else:
    print("No local model found. Running Grid Search...")
    print("Starting Grid Search with Cross-Validation...")
    grid_search.fit(X_train, y_train)
    best_params = grid_search.best_params_
    best_rf = grid_search.best_estimator_
    best_score = grid_search.best_score_




No local model found. Running Grid Search...
Starting Grid Search with Cross-Validation...


2026/04/29 19:00:05 WARNING mlflow.data.pandas_dataset: Failed to infer schema for Pandas dataset. Exception: Expected pandas.Series, got '<class 'pandas.core.frame.DataFrame'>'.


Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [ ]:

# Save locally for next time

joblib.dump(best_rf, MODEL_PATH)
print(f"Grid search complete. Model saved to {MODEL_PATH}")


Grid search complete. Model saved to best_rf_grid_model.pkl


In [ ]:
# 5. Evaluate
print(f"\nBest Parameters: {best_params}")
print(f"Best CV Score: {best_score:.2f}")



Best Parameters: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 5, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 200, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Best CV Score: 0.91


In [ ]:

# Predict using the best found version of the model

y_pred = best_rf.predict(X_test)

In [ ]:


print("\n--- Test Set Metrics ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print(classification_report(y_test, y_pred))


--- Test Set Metrics ---
Accuracy: 0.9109
              precision    recall  f1-score   support

           0       0.91      0.92      0.91     42159
           1       0.92      0.90      0.91     42155

    accuracy                           0.91     84314
   macro avg       0.91      0.91      0.91     84314
weighted avg       0.91      0.91      0.91     84314



In [ ]:
from metrics_utils import calculate_business_metrics
buisness_metrics = calculate_business_metrics(y_test, y_pred, is_holiday_col=IsHoliday)



In [ ]:
mlflow.log_metrics(buisness_metrics)

MlflowException: The run e906d24fc22146d3a111dba4876c00a9 must be in 'active' lifecycle_stage.

In [ ]:
# Get experiment details
exp = mlflow.get_experiment_by_name("Random Forest")
if exp:
    print(f"Experiment ID: {exp.experiment_id}")
    print(f"Artifact Location: {exp.artifact_location}")
    
    # Search for runs in THIS specific experiment
    runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    print(f"Total runs found: {len(runs)}")
    display(runs.head())
else:
    print("Experiment 'quickstart' not found in this mlruns directory.")

Experiment ID: 345543488712145240
Artifact Location: file:///home/sarah/code/forth_year/data_science/project/walmart-sales-classification/src/models/mlruns/345543488712145240
Total runs found: 5


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.std_score_time,metrics.mean_score_time,metrics.rank_test_score,metrics.std_fit_time,...,params.monotonic_cst,params.criterion,tags.mlflow.autologging,tags.mlflow.parentRunId,tags.estimator_name,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.name,tags.estimator_class
0,037665a21022413ba882b08808fe17a1,345543488712145240,FINISHED,file:///home/sarah/code/forth_year/data_scienc...,2026-04-29 15:10:37.129000+00:00,2026-04-29 15:38:02.538000+00:00,1.452949,18.525499,1.0,7.238397,...,None,gini,sklearn,e906d24fc22146d3a111dba4876c00a9,RandomForestClassifier,LOCAL,loud-mule-131,sarah,/home/sarah/.cache/pypoetry/virtualenvs/walmar...,sklearn.ensemble._forest.RandomForestClassifier
1,24de633b48084572a7ff1574df484628,345543488712145240,FINISHED,file:///home/sarah/code/forth_year/data_scienc...,2026-04-29 15:10:37.129000+00:00,2026-04-29 15:38:02.538000+00:00,1.177655,9.503030,2.0,5.111507,...,None,gini,sklearn,e906d24fc22146d3a111dba4876c00a9,RandomForestClassifier,LOCAL,respected-stag-941,sarah,/home/sarah/.cache/pypoetry/virtualenvs/walmar...,sklearn.ensemble._forest.RandomForestClassifier
2,82f3256a0fa443528b2725f1b7d36a28,345543488712145240,FINISHED,file:///home/sarah/code/forth_year/data_scienc...,2026-04-29 15:10:37.129000+00:00,2026-04-29 15:38:02.538000+00:00,0.500206,4.149260,3.0,3.340868,...,None,gini,sklearn,e906d24fc22146d3a111dba4876c00a9,RandomForestClassifier,LOCAL,auspicious-pug-980,sarah,/home/sarah/.cache/pypoetry/virtualenvs/walmar...,sklearn.ensemble._forest.RandomForestClassifier
3,db612ef56f1944378bdb36f1d47a1d39,345543488712145240,FINISHED,file:///home/sarah/code/forth_year/data_scienc...,2026-04-29 15:10:37.129000+00:00,2026-04-29 15:38:02.538000+00:00,1.273467,6.708249,5.0,5.349596,...,None,gini,sklearn,e906d24fc22146d3a111dba4876c00a9,RandomForestClassifier,LOCAL,honorable-stag-745,sarah,/home/sarah/.cache/pypoetry/virtualenvs/walmar...,sklearn.ensemble._forest.RandomForestClassifier
4,eb41992ac04644d2b7be75833ea46f78,345543488712145240,FINISHED,file:///home/sarah/code/forth_year/data_scienc...,2026-04-29 15:10:37.129000+00:00,2026-04-29 15:38:02.538000+00:00,1.170698,5.504894,4.0,3.478532,...,None,gini,sklearn,e906d24fc22146d3a111dba4876c00a9,RandomForestClassifier,LOCAL,grandiose-mule-812,sarah,/home/sarah/.cache/pypoetry/virtualenvs/walmar...,sklearn.ensemble._forest.RandomForestClassifier
